In [1]:
# TODO: informações cstfoil, XFOIL, e prints de single_run.py
# TODO: alterar ftol para 1e-5
# TODO: alterar batentes Al_lower, Al_upper, Au_lower, Au_upper

# Lab 3: Projeto de Perfil

Módulos:
1. eulerblock.exe: airfoil.dat, settings.txt -> wall.dat, solution.vtk, derivatives.dat
2. gridgen2d.py: gera a malha
3. gridwarp.py: deforma a malha mas acho que não faz nada agora
4. euler_mod.py: chama eulerblock.exe
5. airfoil_mod.py (modificado para não usar pip): gera airfoil.dat dos parâmetros

Scripts em eulerblock_run_cases:
1. 01_airfoil_plotter/airfoil_plotter.py: analisar airfoil interativamente
2. 02_single_run/single_run.py: analisar airfoil não-interativamente
3. 03_optimize/optimize.py: otimizar airfoil para uma condição. Pode reinicializar com reload_opt = opt_results.pickle. Parâmetros de otimização: ftol, batentes Al_lower, Al_upper, Au_lower, Au_upper
4. 03_optimize/plot_history.py: plota progresso da otimização
5. paraview_layouts/paraview_state_windows.py: visualizar solution.vtk no paraview

Objetivo: otimizar 2 perfis, um em y=0 e outro em y = 0.9*(b_w/2), para o cruzeiro:
1. Mach = mach_cruise
2. altitude = altitude_cruise
3. Cl = Cl(y): curva elíptica de sustentação no cruzeiro, deformada pela corda variável da asa trapezoidal
4. clmax(y) calculado no XFOIL > clmax_min (evitar stall em qualquer seção da asa)

XFOIL6.99/xfoil.exe é usado pelo pacote AeroSandbox como restrição na otimização e para gerar figuras automaticamente.

Avião usado: Tomav final de PRJ-22

Aerofólio inicial: NASA SC(2)-0714 AIRFOIL (sc20714-il) (http://airfoiltools.com/airfoil/details?airfoil=sc20714-il)

optimize.py:
min cd

w.r.t. CST coefficients(Al, Au), α

s.t. cl = clref

(t/c) ≥ (t/c)ref

Al,lower ⪯ Al ⪯ Al,upper

Au,lower ⪯ Au ⪯ Au,upper

airfoil.dat -> XFOIL: utilize os seguintes comandos para ajustar a geometria:

• load airfoil.dat Carrega o aerofólio

• ppar Menu para ajuste dos painéis

• t Ajustar concentração de painéis no bordo de fuga

• 0.45 Fator de concentração

• n Ajustar número de painéis

• 200 Número de painéis

• <ENTER> Retornar ao menu anterior

• <ENTER> Retornar ao menu anterior

• oper Abrir menu de análise

• iter Definir número de iterações viscosas

• 100 Número máximo de iterações viscosas

• v Ativas análise viscosa

• Re_a ou Re_b Coloque o número de Reynolds desejado

• a Analisar ângulo de ataque

• 5 Angulo de ataque desejado (em graus) 

## Instalar pacotes pip

In [2]:
from importlib.util import find_spec
if find_spec("numpy") is None or find_spec("pandas") is None or find_spec("matplotlib") is None \
    or find_spec("scipy") is None or find_spec("seaborn") is None or find_spec("AeroSandbox") is None:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "pandas", "matplotlib", "scipy", "seaborn", "AeroSandbox"])

## Imports

In [3]:
import sys
import os
import csv
import json
import warnings
import pickle
import time
import shutil
import subprocess
import tempfile
from pathlib import Path
from functools import lru_cache
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import minimize, NonlinearConstraint, minimize_scalar, brentq, Bounds
from scipy import integrate
import aerosandbox as asb

import airfoil_mod as am
from eulerblock import euler_mod as eb

from designTool.aerodynamics import aerodynamics
from designTool.auxiliary import atmosphere
from designTool.balance import balance
from designTool.constants import gravity, ft2m
from designTool.geometry import geometry
from designTool.performance import thrust_matching
from designTool.plots import plot_geometry
from designTool.standard_airplane import standard_airplane
from designTool.weight import weight
from designTool.landing_gear import landing_gear
from designTool.propulsion import engineTSFC
from designTool.analyze import analyze

## Funções

In [4]:
def calcular_CL_voo(condicao, peso):
    atm = atmosphere(condicao["altitude"])
    rho = atm["density"]
    Mach = condicao["Mach"]
    V = Mach*atm["speed_of_sound"]
    CL = 2*peso / (rho*V**2*S_w)
    return CL

def aerodynamics_condicao_e_CL(airplane, condicao, CL):
    CD, CLmax, dragDict = aerodynamics(airplane, condicao["Mach"], condicao["altitude"], CL,
                                       n_engines_failed=condicao["n_engines_failed"],
                                       highlift_config=condicao["highlift_config"], lg_down=condicao["lg_down"],
                                       h_ground=condicao["h_ground"])
    return CD, CLmax, dragDict

def calcular_secao_asa(airplane, condicao, y, W_medio_cruise, log=False):
    # Geometria
    tct_w = airplane["inputs"]["tct_w"]
    tcr_w = airplane["inputs"]["tcr_w"]
    b_w = float(airplane["geometry"]["b_w"])
    cr_w = float(airplane["geometry"]["cr_w"])
    ct_w = float(airplane["geometry"]["ct_w"])
    tc_y = np.interp(y, [0, b_w/2], [tcr_w, tct_w])
    c_y = np.interp(y, [0, b_w/2], [cr_w, ct_w])

    # Aerodinâmica
    Mach = condicao["Mach"]
    altitude = condicao["altitude"]
    atm = atmosphere(altitude)
    rho = atm["density"]
    mu = atm["dyn_viscosity"]
    V_som = atm["speed_of_sound"]
    V = Mach*V_som
    Re = rho*V*c_y/mu

    # cl(y)
    # Distribuição elíptica de sustentação:
    # l(y) = l0 * sqrt(1 - (2y/b_w)^2)
    # Sustentação do perfil:
    # l(y) = q * c(y) * cl(y) (q = rho*V^2/2)
    # Sustentação total da asa: L = q*S_w*CL = W
    # 2 * integral(l(y), 0, b_w/2) = L
    # 2 * integral(l0 * sqrt(1 - (2y/b_w)^2), 0, b_w/2) = L
    # 2 * l0 * (pi*b_w/8) = L
    # l0 * pi * b_w / 4 = L
    # l0 = 4*L/(pi*b_w)
    q = rho*V**2/2
    L = W_medio_cruise
    l0 = 4*L/(np.pi*b_w)
    l_y = l0 * np.sqrt(1 - (2*y/b_w)**2)
    cl_y = l_y/(q*c_y)

    # Print
    if log:
        print(f"  y = {y:.2f} m")
        print(f"  t_y = {(tc_y*100):.2f} cm")
        print(f"  c_y = {c_y:.2f} m")
        print(f"  Re = {Re:.2e}")
        print(f"  cl_y = {cl_y:.4f}")

    return tc_y, c_y, Re, cl_y


def ajustar_tabela_latex(latex):
    latex = latex.replace(r"\toprule", "")
    latex = latex.replace(r"\midrule", r"\hline")
    latex = latex.replace(r"\bottomrule", "")
    latex = latex.replace(r"\hline \\", r"\hline")
    latex = latex.replace("%", r"\%")
    return latex

## Atividades

### 1. Primeiramente precisamos determinar um ponto de projeto (altitude, Mach e peso) para otimizar o aerofólio. Uma vez que esse ponto for estabelecido, calcule o coeficiente de sustentação (CL) correspondente para a aeronave. Em seguida, tome a corda média aerodinâmica da asa como corda de referência (cref) e, para fins desse exercício, assuma que o coeficiente de sustentação dessa seção é igual ao CL da aeronave (clref = CL). Utilize a espessura média da aeronave da equipe para determinar o valor de espessura mínima para a otimização (t/c)ref. Utilize tais dados para preencher a Tab. 1. Mostre no relatório o procedimento de cálculo dos itens presentes na planilha.


#### Avião Tomav no cruzeiro

In [5]:
airplane_name = "Tomav"
airplane = standard_airplane(airplane_name)
analyze(airplane)

W0 = float(airplane["thrust_matching"]["W0"])
b_w = float(airplane["geometry"]["b_w"])
S_w = airplane["inputs"]["S_w"]
clmax_w = airplane["inputs"]["clmax_w"]
Mach_cruise = airplane["inputs"]["Mach_cruise"]
altitude_cruise = airplane["inputs"]["altitude_cruise"]

CondicaoCruise = {
    "Mach": Mach_cruise,
    "altitude": altitude_cruise,
    "n_engines_failed": 0,
    "highlift_config": "clean",
    "lg_down": 0,
    "h_ground": 0,
}

print(f"Mach_cruise = {Mach_cruise:.2f}")
print(f"altitude_cruise = {altitude_cruise:.0f} m")

Mach_cruise = 0.85
altitude_cruise = 9876 m


#### Peso médio no cruzeiro

In [6]:
# Equação de Breguet: R = (V/TSFC)*(L/D)*ln(Wi/Wf)
# => Wf = Wi * exp(-K*R), K = 1/((V/TSFC)*(L/D))
# W_medio = W(x=R/2)
# W_medio = Wi * exp(-K*R/2)
# W_medio = Wi * sqrt(exp(-K*R))
# W_medio = Wi * sqrt(Wf/Wi)
# W_medio = sqrt(Wi * Wf) 
# => média geométrica

fases = ["start", "taxi", "takeoff", "climb", "cruise", "descent", "altcruise", "loiter", "landing"]
fracoes = [airplane["fuel_weight"]["Mf_hist"][fase] for fase in fases]
pesos = [0 for _ in range(len(fracoes) + 1)]
pesos[0] = W0
for i, f in enumerate(fracoes):
    pesos[i + 1] = pesos[i] * fracoes[i]
PesosFinais = {fase: peso for fase, peso in zip(fases, pesos[1:])}
PesosIniciais = {fase: peso for fase, peso in zip(fases, pesos[:-1])}

W_medio_cruise = (PesosIniciais["cruise"]*PesosFinais["cruise"])**0.5
print(f"W_medio_cruise = {(W_medio_cruise/gravity):.1f} kgf")

W_medio_cruise = 204102.9 kgf


#### Aerodinâmica média no cruzeiro

In [7]:
CL_medio_cruise = calcular_CL_voo(CondicaoCruise, W_medio_cruise)
CD_medio_cruise, CLmax_medio_cruise, dragDict_medio_cruise = aerodynamics_condicao_e_CL(airplane, CondicaoCruise, CL_medio_cruise)

atm = atmosphere(altitude_cruise)
rho = atm["density"]
V_som = atm["speed_of_sound"]

print(f"CL_medio_cruise = {CL_medio_cruise:.4f}")
print(f"CD_medio_cruise = {CD_medio_cruise:.4f}")
print(f"CLmax_medio_cruise = {CLmax_medio_cruise:.4f}")
print(f"Velocidade do som = {V_som:.2f} m/s")
print(f"Densidade do ar = {rho:.4f} kg/m³")

CL_medio_cruise = 0.4443
CD_medio_cruise = 0.0206
CLmax_medio_cruise = 1.3262
Velocidade do som = 300.04 m/s
Densidade do ar = 0.4199 kg/m³


#### Seções da asa

In [8]:
y_A = 0
y_B = 0.9 * (b_w/2)
y_medio = 0.5 * (b_w/2)

print("* Seção A:")
tc_A, c_A, Re_A, cl_A = calcular_secao_asa(airplane, CondicaoCruise, y_A, W_medio_cruise, log=True)
print("* Seção B:")
tc_B, c_B, Re_B, cl_B = calcular_secao_asa(airplane, CondicaoCruise, y_B, W_medio_cruise, log=True)
print("* Seção na metade:")
calcular_secao_asa(airplane, CondicaoCruise, y_medio, W_medio_cruise, log=True)

* Seção A:
  y = 0.00 m
  t_y = 17.50 cm
  c_y = 9.58 m
  Re = 6.96e+07
  cl_y = 0.3309
* Seção B:
  y = 26.49 m
  t_y = 8.95 cm
  c_y = 2.42 m
  Re = 1.76e+07
  cl_y = 0.5702
* Seção na metade:
  y = 14.72 m
  t_y = 12.75 cm
  c_y = 5.61 m
  Re = 4.07e+07
  cl_y = 0.4899


(0.1275, 5.606119105813882, 40732910.50592061, 0.4899065214807195)

#### Tabela 1

In [9]:
tabela1 = pd.DataFrame({
    "Parâmetro": [
        r"$h$",
        r"$M_\infty$",
        r"$W$",
        r"$S_{\mathrm{ref}}$",
        r"$b_w$",
        r"$\rho_\infty$",
        r"$a_\infty$",
        r"$y_{A}$",
        r"$c_{A}$",
        r"$c_{l,A}$",
        r"$Re_{A}$",
        r"$(t/c)_{A}$",
        r"$y_{B}$",
        r"$c_{B}$",
        r"$c_{l,B}$",
        r"$Re_{B}$",
        r"$(t/c)_{B}$"
    ],
    "Valor": [
        f"{(altitude_cruise/ft2m):.0f} ft",
        f"{Mach_cruise:.2f}",
        f"{W_medio_cruise/gravity:.1f} kgf",
        rf"{S_w:.1f} $\mathrm{{m^2}}$",
        f"{b_w:.2f} m",
        rf"{rho:.3f} $\mathrm{{kg/m^3}}$",
        rf"{V_som:.1f} $\mathrm{{m/s}}$",
        f"{y_A:.2f} m",
        f"{c_A:.2f} m",
        f"{cl_A:.4f}",
        f"{Re_A:.2e}",
        f"{tc_A:.1%}",
        f"{y_B:.2f} m",
        f"{c_B:.2f} m",
        f"{cl_B:.4f}",
        f"{Re_B:.2e}",
        f"{tc_B:.1%}"
    ],
    "Descrição": [
        "Altitude do ponto de projeto",
        "Mach do ponto de projeto",
        "Peso da aeronave no ponto de projeto",
        "Área de referência da aeronave",
        "Envergadura da asa",
        "Densidade do ar no ponto de projeto",
        r"Velocidade do som no ponto de projeto \\ \hline",
        "Posição da Seção A",
        "Corda da seção A",
        "Coeficiente de sustentação da Seção A",
        "Reynolds para a Seção A",
        r"Espessura relativa da Seção A \\ \hline",
        "Posição da Seção B",
        "Corda da seção B",
        "Coeficiente de sustentação da Seção B",
        "Reynolds para a Seção B",
        "Espessura relativa da Seção B"
    ]
})

print("Tabela 1:")
display(tabela1)

Tabela 1:


,Parâmetro,Valor,Descrição
0,$h$,32400 ft,Altitude do ponto de projeto
1,$M_\infty$,0.85,Mach do ponto de projeto
2,$W$,204102.9 kgf,Peso da aeronave no ponto de projeto
3,$S_{\mathrm{ref}}$,330.0 $\mathrm{m^2}$,Área de referência da aeronave
4,$b_w$,58.86 m,Envergadura da asa
5,$\rho_\infty$,0.420 $\mathrm{kg/m^3}$,Densidade do ar no ponto de projeto
6,$a_\infty$,300.0 $\mathrm{m/s}$,Velocidade do som no ponto de projeto \\ \hline
7,$y_{A}$,0.00 m,Posição da Seção A
8,$c_{A}$,9.58 m,Corda da seção A
9,"$c_{l,A}$",0.3309,Coeficiente de sustentação da Seção A


#### LaTeX

In [10]:
latex1 = tabela1.to_latex(
    index=False,
    escape=False,
    na_rep="",
    caption="Dados para as seções A e B",
    label="tab:dados_secoes",
    column_format=r"c|c|l"
)
latex1 = ajustar_tabela_latex(latex1)
print(latex1)

\begin{table}
\caption{Dados para as seções A e B}
\label{tab:dados_secoes}
\begin{tabular}{c|c|l}

Parâmetro & Valor & Descrição \\
\hline
$h$ & 32400 ft & Altitude do ponto de projeto \\
$M_\infty$ & 0.85 & Mach do ponto de projeto \\
$W$ & 204102.9 kgf & Peso da aeronave no ponto de projeto \\
$S_{\mathrm{ref}}$ & 330.0 $\mathrm{m^2}$ & Área de referência da aeronave \\
$b_w$ & 58.86 m & Envergadura da asa \\
$\rho_\infty$ & 0.420 $\mathrm{kg/m^3}$ & Densidade do ar no ponto de projeto \\
$a_\infty$ & 300.0 $\mathrm{m/s}$ & Velocidade do som no ponto de projeto \\ \hline
$y_{A}$ & 0.00 m & Posição da Seção A \\
$c_{A}$ & 9.58 m & Corda da seção A \\
$c_{l,A}$ & 0.3309 & Coeficiente de sustentação da Seção A \\
$Re_{A}$ & 6.96e+07 & Reynolds para a Seção A \\
$(t/c)_{A}$ & 17.5\% & Espessura relativa da Seção A \\ \hline
$y_{B}$ & 26.49 m & Posição da Seção B \\
$c_{B}$ & 2.42 m & Corda da seção B \\
$c_{l,B}$ & 0.5702 & Coeficiente de sustentação da Seção B \\
$Re_{B}$ & 1.76e+07 & 

### 2. Utilize os scripts da pasta 02_single_run para determinar qual o ângulo de ataque (variável alpha da linha 19) faz com que o perfil NACA 1411 atinja o cl de projeto. Lembre de atualizar também o número de Mach.

### 3. Utilize os scripts da pasta 03_optimize para obter um aerofólio otimizado para as condições de projeto. O ponto de partida da otimização será o aerofólio NACA 1411. Utilize como batentes para parâmetros do CST os valores Al,upper = −0.05 e Au,lower = 0.05. Preencha a Tab. 2 com os resultados da otimização. ATENÇÃO: Cada otimização pode demorar horas, por isso guarde bem o resultado.


#### Tabela 2

#### LaTeX

### 4. Plote o histórico de convergência de cada otimização (pode usar o script plot_history.py) e discuta quanto foi o aprimoramento obtido pelo otimizador. Indique também o tempo total de otimização e como as restrições afetaram o resultado.


### 5. Desenhe 1) gráficos sobrepondo a geometria, 2) gráficos sobrepondo curvas de Cp ao longo da corda e 3) gráficos sobrepondo curvas de Mach ao longo da corda dos aerofólios da Tab. 2 na condição de projeto transônica. O script da pasta 02_single_run pode ser útil para isso.

### 6. Use os gráficos gerados para explicar a abordagem aplicada pelo otimizador para chegar na configuração ótima.

### 7. Adapte o script 02_single_run.py para gerar uma polar de arrasto para o perfil otimizado na condição transônica. Para isso, você precisará simular o perfil para uma sequência de ângulos de ataque e depois plotar os valores de cl e cd. Sugiro usar um intervalo de -2/+2 graus em relação ao ângulo de ataque do ponto ótimo. Lembre-se de destacar o ponto de projeto (cl da otimização) na polar. Sobreponha no mesmo gráfico uma curva correspondente ao aerofólio transônico RAE2822 (os dados desse aerofólio estão no arquivo single_run.py). Discuta se a polar do aerofólio otimizado tem alguma peculiaridade e se isso é bom ou ruim para o projeto de uma aeronave.

### 8. Exporte os aerofólios iniciais e finais da otimização para o formato Xfoil e gere curvas cl × α, cl × cd e cm × α para os dois aerofólios em regime subsônico (pode desconsiderar o número de Mach para essa análise). Utilize a condição de decolagem da aeronave para determinar o número de Reynolds para essa simulação (lembre de indicar o número de Reynolds no relatório). Sobreponha as curvas dos dois aerofólios em cada plot e discuta as vantagens e desvantagens dos perfis no regime analisado.

### 9. Verifiquem a possibilidade de alterar a definição da otimização para obter um aerofólio de melhor característica no regime subsônico.

### 10. Quem estiver com o Paraview (https://www.paraview.org/download/) instalado no computador pode observar a solução no domínio carregando o arquivo paraview_state_windows.py com a opção “Load State”. Esse arquivo sempre carregará o arquivo solution.vtk da pasta 02_single_run.